In [ ]:
!pip install torch torchvision matplotlib numpy 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from matplotlib import pyplot as plt
import numpy as np

In [2]:
class Encoder(nn.Module):

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)

        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):

        h = torch.relu(self.fc1(x))

        mu = self.fc_mu(h)

        logvar = self.fc_logvar(h)

        return mu, logvar

In [3]:
class Decoder(nn.Module):

    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()

        self.fc1 = nn.Linear(latent_dim, hidden_dim)

        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):

        h = torch.relu(self.fc1(z))

        x_hat = torch.sigmoid(self.fc2(h))

        return x_hat

In [4]:
class VAE(nn.Module):

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE, self).__init__()

        self.encoder = Encoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim
        )

        self.decoder = Decoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            output_dim=input_dim
        )

    def forward(self, x):

        # Encoder
        mu, logvar = self.encoder(x)

        # Reparameterization trick
        std = torch.exp(0.5 * logvar)

        eps = torch.randn_like(std)

        z = mu + eps * std

        # Decoder
        x_hat = self.decoder(z)

        return x_hat, mu, logvar

In [5]:
def loss_function(x, x_hat, mu, logvar):

    # Reconstruction loss
    BCE = nn.functional.binary_cross_entropy(
        x_hat,
        x,
        reduction='sum'
    )

    # KL divergence
    KLD = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    return BCE + KLD

In [6]:
input_dim = 784
hidden_dim = 400
latent_dim = 20

lr = 1e-3
batch_size = 128
epochs = 10

In [7]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

In [8]:
vae = VAE(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim
)

optimizer = optim.Adam(
    vae.parameters(),
    lr=lr
)

In [9]:
vae.train()

for epoch in range(epochs):

    train_loss = 0

    for x, _ in train_loader:

        # Make sure input is [batch_size, 784]
        x = x.view(-1, input_dim)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        x_hat, mu, logvar = vae(x)

        # Calculate loss
        loss = loss_function(
            x,
            x_hat,
            mu,
            logvar
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        train_loss += loss.item()

    average_loss = train_loss / len(train_loader.dataset)

    print(
        f"Epoch [{epoch + 1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/10], Loss: 164.6568
Epoch [2/10], Loss: 121.8594
Epoch [3/10], Loss: 114.5362
Epoch [4/10], Loss: 111.4232
Epoch [5/10], Loss: 109.6070
Epoch [6/10], Loss: 108.4781
Epoch [7/10], Loss: 107.6527
Epoch [8/10], Loss: 107.0129
Epoch [9/10], Loss: 106.5023
Epoch [10/10], Loss: 106.0677


In [ ]:
# Testing and evaluating the model

vae.eval()

with torch.no_grad():

    # Get one batch of images
    x, _ = next(iter(train_loader))

    # Flatten images
    x = x.view(-1, input_dim)

    # Reconstruct images
    x_hat, _, _ = vae(x)

    # Convert back to 28 x 28
    x = x.view(-1, 28, 28)
    x_hat = x_hat.view(-1, 28, 28)

    # Create figure
    fig, axs = plt.subplots(2, 10, figsize=(15, 3))

    for i in range(10):

        # Original image
        axs[0, i].imshow(
            x[i].cpu().numpy(),
            cmap='gray'
        )
        axs[0, i].axis('off')

        # Reconstructed image
        axs[1, i].imshow(
            x_hat[i].cpu().numpy(),
            cmap='gray'
        )
        axs[1, i].axis('off')

    plt.show()